# [Introduction to Data Science](http://datascience-intro.github.io/1MS041-2026/)    
## 1MS041, 2026 
&copy;2026 Raazesh Sainudiin, Benny Avelin. [Attribution 4.0 International     (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/)

# Lecture 15: checking regression models and their error bounds

This scheduled notebook revisits Sections 9.4--9.5: regression
metrics, bounded squared loss, the edge cases of $R^2$ and FVU, and
Hoeffding versus Bennett bounds. Data are generated locally or bundled
with scikit-learn, so no network access is required.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(2026)
np.set_printoptions(precision=4, suppress=True)


## Measuring regression error on test data

On an independent test sample,
$$
\operatorname{MSE}=m^{-1}\sum_i(y_i-\widehat y_i)^2,\qquad
\operatorname{MAE}=m^{-1}\sum_i|y_i-\widehat y_i|.
$$
If $\operatorname{SST}=\sum_i(y_i-\bar y)^2>0$, then
$$
\operatorname{FVU}=\operatorname{SSE}/\operatorname{SST},\qquad
R^2=1-\operatorname{FVU}.
$$
These are test summaries. The notation $R^2$ is not a square, and
its value may be negative.


In [ ]:
def regression_metrics(y_true, y_pred, atol=1e-14):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    residual = y_true - y_pred
    mse, mae = np.mean(residual**2), np.mean(np.abs(residual))
    sse = np.sum(residual**2)
    sst = np.sum((y_true - y_true.mean()) ** 2)
    if sst <= atol:
        fvu, r_squared = np.nan, np.nan
    else:
        fvu, r_squared = sse / sst, 1 - sse / sst
    return {"mse": mse, "mae": mae, "sse": sse, "sst": sst,
            "fvu": fvu, "r2": r_squared}

diabetes = load_diabetes()
X_train, X_test, y_train, y_test = train_test_split(
    diabetes.data, diabetes.target, test_size=0.25, random_state=2026
)
model = make_pipeline(StandardScaler(), Ridge(alpha=10))
model.fit(X_train, y_train)
prediction = model.predict(X_test)
heldout_metrics = regression_metrics(y_test, prediction)
print(heldout_metrics)

plt.scatter(prediction, y_test, alpha=0.65)
limits = [min(prediction.min(), y_test.min()),
          max(prediction.max(), y_test.max())]
plt.plot(limits, limits, "k--")
plt.xlabel("held-out prediction")
plt.ylabel("held-out response")
plt.show()


The diabetes population is not assumed bounded. Its observed minimum
and maximum cannot justify a bounded-loss concentration result.

## When $R^2$ and FVU need care

When all test responses are equal, $\operatorname{SST}=0$, so both
ratios are undefined. A predictor worse than the test-mean benchmark
can have FVU above one and negative $R^2$.


In [ ]:
print("constant response:",
      regression_metrics([3, 3, 3, 3], [2, 3, 4, 5]))
print("negative R^2:",
      regression_metrics([-1, 0, 1], [10, 10, 10]))


## A confidence bound for bounded loss

Specify $X\sim\mathrm{Uniform}[-1,1]$,
$E\sim\mathrm{Uniform}[-0.2,0.2]$, independently, and
$Y=0.75X+E$. Thus $|Y|<1$. Fit on one sample, clip predictions to
$[-1,1]$, and evaluate on an independent sample. Squared losses then
lie in $[0,4]$ almost surely, not merely in the observed sample.


In [ ]:
def draw_bounded_regression(n, rng):
    x = rng.uniform(-1, 1, size=(n, 1))
    error = rng.uniform(-0.2, 0.2, size=n)
    return x, 0.75 * x[:, 0] + error

def hoeffding_radius(sample_size, B, alpha=0.05):
    return B * np.sqrt(np.log(2 / alpha) / (2 * sample_size))

Xb_train, yb_train = draw_bounded_regression(500, rng)
Xb_test, yb_test = draw_bounded_regression(300, rng)
bounded_model = LinearRegression().fit(Xb_train, yb_train)
bounded_prediction = np.clip(bounded_model.predict(Xb_test), -1, 1)
losses = (yb_test - bounded_prediction) ** 2
B_loss = 4
radius = hoeffding_radius(len(losses), B_loss)
interval = max(0, losses.mean() - radius), min(B_loss, losses.mean() + radius)
print("maximum observed loss; valid bound:", losses.max(), B_loss)
print("test MSE and 95% Hoeffding interval:", losses.mean(), interval)


Hoeffding uses only the range and may be conservative. Its
interpretation is clean because, after training is fixed, the test
losses are i.i.d. in a known interval.

## Using variance in Bennett's inequality

For i.i.d. losses with mean $\mu$, population variance $v>0$, and
$W_i-\mu\leq b$, Bennett gives
$$
\mathbb P(\bar W_m-\mu\geq\epsilon)
\leq\exp\left[-\frac{mv}{b^2}
h\left(\frac{b\epsilon}{v}\right)\right],
\quad h(u)=(1+u)\log(1+u)-u.
$$
For losses in $[0,B]$, use $b=B$ on each tail and a union bound.
The variance is a population quantity. Substituting the variance
computed from the same test set is not justified by ordinary Bennett.


In [ ]:
def bennett_h(u):
    return (1 + u) * np.log1p(u) - u

def bennett_radius(sample_size, B, population_variance, alpha=0.05):
    if population_variance == 0:
        return 0.0
    target = np.log(2 / alpha)
    def exponent(epsilon):
        u = B * epsilon / population_variance
        return (sample_size * population_variance / B**2) * bennett_h(u)
    low, high = 0.0, B
    if exponent(high) < target:
        return B
    for _ in range(80):
        middle = (low + high) / 2
        if exponent(middle) < target:
            low = middle
        else:
            high = middle
    return high

# Regression with Y~Bernoulli(p), fixed predictor g=0:
# squared loss W=Y has known mean p and variance p(1-p).
p, m = 0.05, 400
W = rng.binomial(1, p, size=m).astype(float)
population_variance = p * (1 - p)
hoeffding_eps = hoeffding_radius(m, B=1)
bennett_eps = bennett_radius(m, B=1,
                             population_variance=population_variance)
print("observed mean loss/true risk:", W.mean(), p)
print("95% Hoeffding radius:", hoeffding_eps)
print("95% Bennett radius:", bennett_eps)
print("sample variance, diagnostic only:", W.var(ddof=1))


Bennett is sharper here because the known variance $p(1-p)$ is
small relative to the full range. If population variance is unknown,
an empirical Bernstein or empirical Bennett theorem with its own
correction is required.

## Try it yourself

1. Construct predictions with FVU above one, and a constant-response
   case where both ratios are undefined.
2. Derive $0\leq(Y-g(X))^2\leq4M^2$ from
   $|Y|,|g(X)|\leq M$, and calculate a test size for a chosen
   Hoeffding half-width.
3. Vary $p$ and $m$ in the Bernoulli-loss example. Plot both
   radii and state exactly which variance must be known for ordinary
   Bennett.
